# Step 2. Data Understanding

Step 1 framed the problem. Step 2 checks that the data can carry it, and builds the reference every later step reads from.

Three questions guide it.

1. Where does the data come from, and can it be trusted?
2. What is in the table, and is it clean?
3. What kind of thing is each column, and which columns matter for fairness?

The work here feeds Step 3, which cleans and prepares the data, and Step 5, which audits it for bias.

In [1]:
import sys
from pathlib import Path

# I add the repo root to the path so src/ is importable from notebooks/.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
from src.data import (
    load_raw, load_binary,
    CONTINUOUS, BINARY_FLAGS, NOMINAL_CODED, COUNT_ORDINAL, LEAKAGE,
    SENSITIVE, TARGET,
)

df = load_raw()      # all 37 columns, names cleaned
bdf = load_binary()  # Dropout vs Graduate, with a 0/1 dropout column
print("raw", df.shape, "| binary", bdf.shape)

raw (4424, 37) | binary (3630, 38)


## 1. Where the Data Comes From

The data is the UCI set Predict Students' Dropout and Academic Success, from Realinho and colleagues at the Polytechnic Institute of Portalegre. It carries a CC BY 4.0 license, free to reuse with credit, under DOI 10.24432/C5MC89. The full citation, funding, and license sit in data/README.md.

The file gathers several separate institutional databases into one table, one row per student. The providers state they cleaned it for anomalies, outliers, and missing values before release. That matters for the next section. The clean state here is their work, not a property of raw institutional records.

## 2. Dataset Overview

A first pass over the whole table. Size, column types, missing values, duplicate rows, the outcome split, and the spread of the true numbers.

In [2]:
print("shape", df.shape)
print()
print("dtypes")
print(df.dtypes.value_counts())
print()
print("missing values total", int(df.isna().sum().sum()))
print("duplicate rows", int(df.duplicated().sum()))
print()
print("target split")
print(df[TARGET].value_counts())

shape (4424, 37)

dtypes
int64      29
float64     7
str         1
Name: count, dtype: int64

missing values total 0
duplicate rows 0

target split
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


In [3]:
# I check the ranges and spread of the six true numbers. Order and distance are real here.
df[CONTINUOUS].describe().round(1)

,Previous qualification (grade),Admission grade,Age at enrollment,Unemployment rate,Inflation rate,GDP
count,4424.0,4424.0,4424.0,4424.0,4424.0,4424.0
mean,132.6,127.0,23.3,11.6,1.2,0.0
std,13.2,14.5,7.6,2.7,1.4,2.3
min,95.0,95.0,17.0,7.6,-0.8,-4.1
25%,125.0,117.9,19.0,9.4,0.3,-1.7
50%,133.1,126.1,20.0,11.1,1.4,0.3
75%,140.0,134.8,25.0,13.9,2.6,1.8
max,190.0,190.0,70.0,16.2,3.7,3.5


No missing cells and no duplicate rows. This is the providers' cleaning noted above, not luck, so Step 3 validates ranges and keeps real extremes like mature student ages, rather than re-cleaning. The outcome splits three ways, Graduate 2209, Dropout 1421, Enrolled 794. The binary frame drops Enrolled and works with 3630 students. The ranges look sound, with grades between 0 and 200 and ages that reach into mature entry.

## 3. Feature Types

The 37 columns are not one kind of thing. They fall into a few types, and each type needs different handling later. Naming the types now prevents a common mistake in Step 3.

Four types describe the model inputs.

True numbers. Six columns where the value is a real quantity and order and distance mean something, like admission grade and age.

Yes or no flags. Eight columns stored as 1 or 0, like gender, scholarship holder, and debtor.

Coded categories. Nine columns where an integer stands for a category, like course and mother's qualification. The number is a label, not an amount. A course coded 9500 is not greater than one coded 33, it is a different course. Treating these as numbers, scaling them or measuring plain correlation on them, is wrong, so Step 3 handles them as categories.

Ordered count. One column, application order, a rank from 0 first choice to 9 last choice.

Beyond these sit twelve curricular columns from the first and second semesters, held out as leakage per Step 1. The Target column is the outcome.

In [4]:
# I confirm the grouping covers all 37 columns.
groups = {
    "true number": CONTINUOUS,
    "yes or no flag": BINARY_FLAGS,
    "coded category": NOMINAL_CODED,
    "ordered count": COUNT_ORDINAL,
    "curricular (leakage)": LEAKAGE,
    "target": [TARGET],
}
for name, cols in groups.items():
    print(f"{name:22} {len(cols)}")
print("total", sum(len(c) for c in groups.values()))

true number            6
yes or no flag         8
coded category         9
ordered count          1
curricular (leakage)   12
target                 1
total 37


## 4. The Data Dictionary

The dictionary lists every column with its type, its range or values, its unique count, and whether it is a sensitive attribute. Built from the file in the next cell, it stays true to the data and cannot drift from it. A grouped, plain language version follows for reading.

In [5]:
# I build the dictionary from the data, so it matches the file exactly.
group_lookup = {c: name for name, cols in groups.items() for c in cols}

def describe_range(col):
    s = df[col]
    if col in CONTINUOUS or col in LEAKAGE:
        return f"{s.min():g} to {s.max():g}"
    if col in BINARY_FLAGS:
        return "0 or 1"
    if col in NOMINAL_CODED:
        return f"{s.nunique()} codes"
    if col in COUNT_ORDINAL:
        return f"{int(s.min())} to {int(s.max())}"
    if col == TARGET:
        return ", ".join(map(str, s.unique()))
    return ""

data_dict = pd.DataFrame([
    {
        "column": c,
        "group": group_lookup.get(c, "other"),
        "dtype": str(df[c].dtype),
        "unique": int(df[c].nunique()),
        "range or values": describe_range(c),
        "missing": int(df[c].isna().sum()),
        "sensitive": "yes" if c in SENSITIVE else "",
    }
    for c in df.columns
])
data_dict

,column,group,dtype,unique,range or values,missing,sensitive
0,Marital status,coded category,int64,6,6 codes,0,
1,Application mode,coded category,int64,18,18 codes,0,
2,Application order,ordered count,int64,8,0 to 9,0,
3,Course,coded category,int64,17,17 codes,0,
4,Daytime/evening attendance,yes or no flag,int64,2,0 or 1,0,
5,Previous qualification,coded category,int64,17,17 codes,0,
6,Previous qualification (grade),true number,float64,101,95 to 190,0,
7,Nacionality,coded category,int64,21,21 codes,0,
8,Mother's qualification,coded category,int64,29,29 codes,0,
9,Father's qualification,coded category,int64,34,34 codes,0,


Here are the same columns grouped, with plain meaning. The table above holds the exact types and ranges.

| group | columns | meaning |
|---|---|---|
| true number | Admission grade, Previous qualification (grade), Age at enrollment, Unemployment rate, Inflation rate, GDP | real quantities where order and distance matter |
| yes or no flag | Gender, Scholarship holder, Debtor, Tuition fees up to date, Displaced, Educational special needs, International, Daytime/evening attendance | stored as 1 or 0 |
| coded category | Course, Marital status, Application mode, Previous qualification, Nacionality, Mother's and Father's qualification, Mother's and Father's occupation | integer codes standing for categories, labels not amounts |
| ordered count | Application order | rank from 0 first choice to 9 last choice |
| curricular, leakage | twelve first and second semester records | performance after enrollment, held out of the model |
| target | Target | Dropout, Graduate, Enrolled, with Enrolled dropped for the binary frame |

The sensitive attributes for the Step 5 audit are Gender, Age at enrollment, Scholarship holder, and Debtor. Tuition fees up to date is watched too, for the reason the next section shows.

## 5. Sensitive Attributes, and a First Look at Disparity

Four attributes drive the fairness work in Step 5. Gender, age at enrollment, scholarship holder, and debtor. The next cell shows the dropout rate inside each group, against an overall rate near 39 percent. This is the baseline the Step 5 audit builds on.

In [6]:
bdf["age band"] = pd.cut(
    bdf["Age at enrollment"], bins=[16, 20, 23, 30, 100],
    labels=["17 to 20", "21 to 23", "24 to 30", "31 plus"],
)

overall = bdf["dropout"].mean()
print("overall dropout rate", round(overall * 100, 1), "percent")
print()

def show_rate(col, labels=None):
    g = bdf.groupby(col, observed=True)["dropout"].agg(["mean", "size"])
    print(col)
    for idx, row in g.iterrows():
        name = labels.get(idx, idx) if labels else idx
        print(f"  {str(name):16} {row['mean'] * 100:5.1f} percent   n={int(row['size'])}")
    print()

show_rate("Gender", {1: "male", 0: "female"})
show_rate("Scholarship holder", {1: "scholarship", 0: "no scholarship"})
show_rate("Debtor", {1: "debtor", 0: "not debtor"})
show_rate("Tuition fees up to date", {1: "up to date", 0: "not up to date"})
show_rate("age band")

overall dropout rate 39.1 percent

Gender
  female            30.2 percent   n=2381
  male              56.1 percent   n=1249

Scholarship holder
  no scholarship    48.4 percent   n=2661
  scholarship       13.8 percent   n=969

Debtor
  not debtor        34.5 percent   n=3217
  debtor            75.5 percent   n=413

Tuition fees up to date
  not up to date    94.0 percent   n=486
  up to date        30.7 percent   n=3144

age band
  17 to 20          26.1 percent   n=2080
  21 to 23          40.6 percent   n=473
  24 to 30          66.5 percent   n=499
  31 plus           61.4 percent   n=578



### Reading the Gaps

The gaps are real and large. Men drop out far more than women, 56 against 30 percent. Students without a scholarship drop out far more than holders, 48 against 14. Students with debt drop out far more than those without, 76 against 35. Older entrants drop out more than the youngest, near 66 percent in the 24 to 30 band against 26 in the 17 to 20 band. A model trained here can carry these gaps into its flags, so Step 5 measures them and works to reduce them.

One column stands apart, tuition fees up to date. Students not up to date drop out almost always, 94 percent. This is not a normal background fact. A student who has stopped paying is often a student already leaving, so this flag sits close to the outcome, like the semester records held out in Step 1. It stays for now, watched. Step 3 tests the model with and without it, so its heavy influence is measured rather than hidden.

## What Step 2 Settles

Five things carry into Step 3.

1. The dataset is cited and licensed, with the full record in data/README.md.
2. There are no missing values and no duplicate rows, a clean state the providers curated.
3. The 37 columns fall into four input types, plus twelve curricular columns held out as leakage.
4. The coded categories are labels, not amounts, so they need category handling, not scaling.
5. All four sensitive attributes show real disparity, and tuition status behaves like a near-outcome signal to watch.

Step 3 cleans, explores, and engineers features on this base, routing each feature type the correct way.